# Playbook: Inspect a GTFS-Realtime Feed

Pulls MBTA's live vehicle-positions feed via `shared.gtfs_rt`, shows the raw table, caches a snapshot, and plots every vehicle on a dark basemap. Swap the agency/feed args (see `shared/agencies.yaml`) to point this at a different one.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import folium

from shared.gtfs_rt import get_single_pull

## Pull the feed

In [ ]:
df = get_single_pull("mbta", "vehicle_positions")
print(f"{len(df)} vehicles")
df.head()

## Cache the raw snapshot

In [ ]:
pulled_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
raw_dir = Path(f"data/raw/vehicle_positions/{pulled_at}")
raw_dir.mkdir(parents=True, exist_ok=True)
df.to_parquet(raw_dir / "data.parquet", index=False)

## Map every vehicle

In [ ]:
map_center = [df["lat"].mean(), df["lon"].mean()]
m = folium.Map(location=map_center, zoom_start=12, tiles="CartoDB dark_matter")

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        color="#2A6F97",
        fill=True,
        fill_opacity=0.85,
        popup=f"{row['route_id']} \u00b7 vehicle {row['vehicle_id']}",
    ).add_to(m)

output_dir = Path(f"outputs/{datetime.now(timezone.utc).date()}/charts")
output_dir.mkdir(parents=True, exist_ok=True)
m.save(output_dir / "vehicle_positions_map.html")
m

*Source: MBTA GTFS-Realtime*